# Bronze CDC Ingestion

## Import Helper Functions

In [1]:
from src import *

# while it is allowed to use the import above, it is advised to list out what we have imported
# from src.config_loader import load_config
# from src.spark_sql_magic import sql
# from src.avro_utils import fetch_avro_schema

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
BRONZE_NAMESPACE = cfg["general"]["namespaces"]["bronze"]
CHECKPOINT_BASE = cfg["kafka"]["checkpoint_base"]
KAFKA_BOOTSTRAP = cfg["kafka"]["bootstrap_servers"]
TOPIC_PREFIX = cfg["kafka"]["topic_prefix"]
TOPIC_SUFFIX = set(cfg["kafka"]["topic_suffix"])
APICURIO_URL = cfg["api"]["apicurio_url"]

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["bronze_cdc_ingestion_backfill"])
        .getOrCreate()
)

## Function: Ingest Topic to Bronze

In [4]:
from pyspark.sql.avro.functions import from_avro

def ingest_topic_to_bronze(topic: str):

    table_name = f"{CATALOG}.{BRONZE_NAMESPACE}.{topic}"
    audit_table = f"{CATALOG}.{BRONZE_NAMESPACE}_audit.{topic}_audit"
    checkpoint_path = f"{CHECKPOINT_BASE}/{BRONZE_NAMESPACE}/{topic}"

    print(f"Starting Bronze ingestion for {topic}")

    # 1. Fetch Schema
    schema_dict = fetch_avro_schema(topic, APICURIO_URL, TOPIC_PREFIX)
    schema_json = json.dumps(schema_dict)

    # 2. Read Kafka stream (for 1st run/backfill pls change to earliest, otherwise latest)
    kafka_df = (
        spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
            .option("subscribe", f"{TOPIC_PREFIX}.{topic}")
            .option("startingOffsets", "latest")
            .load()
    )

    # 3. Use Kafka Stream and Parse Avro
    parsed_df = (
        kafka_df
            .select(
                from_avro("value", schema_json).alias("data"),
                F.col("offset").alias("kafka_offset"),
                F.col("partition").alias("kafka_partition"),
                F.col("timestamp").alias("kafka_ingest_ts")
            )
            .select("data.*", "kafka_offset", "kafka_partition", "kafka_ingest_ts")
            .withColumn("spark_ingest_ts", F.current_timestamp())
    )

    # 4. foreachBatch writer
    def write_batch(batch_df, batch_id):
        if batch_df.isEmpty():
            return

        batch_df = batch_df.withColumn("batch_id", F.lit(str(batch_id)))

        # --- Write bronze table ---     
        batch_df.write.format("iceberg").mode("append").option("mergeSchema", "true").saveAsTable(table_name)

        ## --- audit per partition ---
        audit_partition_df  = (
            batch_df
            .groupBy("kafka_partition")
            .agg(
                F.lit(f"{TOPIC_PREFIX}.{topic}").alias("topic"),
                F.lit(str(batch_id)).alias("batch_id"),
                F.current_timestamp().alias("batch_ts"),
                F.min("kafka_offset").alias("min_offset"),
                F.max("kafka_offset").alias("max_offset"),
                F.count("*").alias("row_count")
            )
        )

        audit_partition_df.write.format("iceberg").mode("append").saveAsTable(audit_table)

        # --- Print total row count ---
        total_rows = audit_partition_df.agg(F.sum("row_count")).collect()[0][0]
        print(f"Batch {batch_id} written with {total_rows} rows across {audit_partition_df.count()} partitions")

    # 5. Write to Iceberg Bronze
    return (
        parsed_df.writeStream
            .foreachBatch(write_batch)
            .option("checkpointLocation", checkpoint_path)
            .trigger(availableNow=True)  # Databricks-friendly
            .start()
    )


## Start All Topic Streams

In [5]:
queries = []

for topic in TOPIC_SUFFIX:
    q = ingest_topic_to_bronze(topic)
    queries.append(q)

for q in queries:
    q.awaitTermination()

Starting Bronze ingestion for products
Starting Bronze ingestion for sellers
Starting Bronze ingestion for customers
Starting Bronze ingestion for order_payments
Starting Bronze ingestion for order_items
Starting Bronze ingestion for order_reviews
Starting Bronze ingestion for orders


## Sanity Check Show All Tables

In [6]:
%%sql
SHOW TABLES IN bronze;

+---------+--------------+-----------+
|namespace|tableName     |isTemporary|
+---------+--------------+-----------+
|bronze   |customers     |false      |
|bronze   |orders        |false      |
|bronze   |order_reviews |false      |
|bronze   |order_payments|false      |
|bronze   |sellers       |false      |
|bronze   |products      |false      |
|bronze   |order_items   |false      |
+---------+--------------+-----------+



In [7]:
%%sql
SHOW TABLES IN bronze_audit;

+------------+--------------------+-----------+
|namespace   |tableName           |isTemporary|
+------------+--------------------+-----------+
|bronze_audit|orders_audit        |false      |
|bronze_audit|customers_audit     |false      |
|bronze_audit|order_reviews_audit |false      |
|bronze_audit|products_audit      |false      |
|bronze_audit|sellers_audit       |false      |
|bronze_audit|order_payments_audit|false      |
|bronze_audit|order_items_audit   |false      |
+------------+--------------------+-----------+



In [8]:
for item in TOPIC_SUFFIX:
    df = spark.table(f"polaris.bronze.{item}").limit(5)
    df.show()
    df = spark.table(f"polaris.bronze_audit.{item}_audit").limit(5)
    df.show()

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------+----------------+----+-------------+------------+---------------+--------------------+--------+--------------------+
|          product_id|product_category_name|product_name_length|product_description_length|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|      created_at|      updated_at|__op|      __ts_ms|kafka_offset|kafka_partition|     kafka_ingest_ts|batch_id|     spark_ingest_ts|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------+----------------+----+-------------+------------+---------------+--------------------+--------+--------------------+
|2c715979f8471e42e...|     moveis_decoracao|              

In [9]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop()  